In [0]:
# Databricks notebook source
# Silver - Squad 3 - ecommerce_clientes
# Fluxo: Bronze Delta -> Silver Delta

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
BRONZE_TABLE = "ecommerce_clientes"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "ecommerce_clientes"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

SILVER_WRITE_MODE = "overwrite"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze.printSchema()
display(df_bronze.limit(10))

In [0]:
# criar DataFrame Silver com limpeza, tipagem e metadado de processamento

from pyspark.sql.functions import col, trim, lower, to_timestamp, current_timestamp

df_silver = (
    df_bronze
    .select(
        col("id_cliente").cast("int").alias("id_cliente"),
        col("uuid_cliente").cast("string").alias("uuid_cliente"),
        trim(col("nome")).alias("nome"),
        trim(col("sobrenome")).alias("sobrenome"),
        lower(trim(col("email"))).alias("email"),
        col("senha_hash").cast("string").alias("senha_hash"),
        to_timestamp(col("dt_cadastro")).alias("dt_cadastro"),
        to_timestamp(col("dt_ultima_atualizacao")).alias("dt_ultima_atualizacao"),

        # metadados herdados da Bronze
        col("bronze_ingested_at").cast("timestamp").alias("bronze_ingested_at"),
        col("bronze_source_file").cast("string").alias("bronze_source_file"),

        # metadado da Silver
        current_timestamp().alias("silver_processed_at"),

        # colunas de particionamento
        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

In [0]:
df_silver.printSchema()

display(df_silver.limit(10))

In [0]:
from pyspark.sql.functions import count, when, col

display(
    df_silver.select(
        count("*").alias("total_linhas"),
        count(when(col("id_cliente").isNull(), True)).alias("id_cliente_nulo"),
        count(when(col("dt_cadastro").isNull(), True)).alias("dt_cadastro_nulo"),
        count(when(col("dt_ultima_atualizacao").isNull(), True)).alias("dt_ultima_atualizacao_nulo"),
        count(when(col("ano").isNull(), True)).alias("ano_nulo"),
        count(when(col("mes").isNull(), True)).alias("mes_nulo"),
        count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo")
    )
)

In [0]:
from pyspark.sql.functions import count as spark_count

df_duplicados = (
    df_silver
    .groupBy("id_cliente")
    .agg(spark_count("*").alias("qtd_registros"))
    .filter(col("qtd_registros") > 1)
)

display(df_duplicados)

In [0]:
(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Dados gravados com sucesso na Silver: {SILVER_PATH}")

In [0]:
df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver_saved.printSchema()

total_silver = df_silver_saved.count()

print(f"Total de registros na Silver: {total_silver}")

display(df_silver_saved.limit(10))

In [0]:
# validar quantidade Bronze x Silver

total_bronze = df_bronze.count()
total_silver = df_silver_saved.count()

print(f"Total Bronze: {total_bronze}")
print(f"Total Silver: {total_silver}")

if total_bronze == total_silver:
    print("Validação OK: Bronze e Silver têm a mesma quantidade de registros.")
else:
    print("Atenção: Bronze e Silver têm quantidades diferentes.")

In [0]:
display(
    df_silver_saved
    .select("ano", "mes")
    .distinct()
    .orderBy("ano", "mes")
)